<center><img src='https://raw.githubusercontent.com/Jangrae/img/master/ml_python.png' width=600/></center>

<img src = "https://github.com/Jangrae/img/blob/master/medical2.png?raw=true" width=800 align="left"/>

# 실습내용

- 학습용 데이터와 평가용 데이터가 별도의 파일로 제공되는 경우가 있습니다.
- 이미 우리가 학습한 방법과는 조금 다른 접근이 필요합니다.
- 의료비 예측을 의뢰한 가상 고객의 요청을 처리해 봅니다.

# 1.환경 준비

- 기본 라이브러리와 대상 데이터를 가져와 이후 과정을 준비합니다.

In [2]:
# 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings(action='ignore')
%config InlineBackend.figure_format = 'retina'

In [3]:
# 데이터 불러오기
path = 'https://raw.githubusercontent.com/Jangrae/csv/master/insurance_train.csv'
data = pd.read_csv(path)

# 2.데이터 이해

- 분석할 데이터를 충분히 이해할 수 있도록 다양한 탐색 과정을 수행합니다.
- 모델링에 집중하기 위해 본 실습에서는 탐색 과정을 생략합니다.

In [4]:
# 데이터 확인
data

# charges: 회원들에게 지출한 의료비

,age,sex,bmi,children,smoker,region,charges
0,41,female,31.600,0,no,southwest,6186.1270
1,30,male,25.460,0,no,northeast,3645.0894
2,18,female,30.115,0,no,northeast,21344.8467
3,61,female,29.920,3,yes,southeast,30942.1918
4,34,female,27.500,1,no,southwest,5003.8530
...,...,...,...,...,...,...,...
1233,50,male,30.970,3,no,northwest,10600.5483
1234,18,female,31.920,0,no,northeast,2205.9808
1235,18,female,36.850,0,no,southeast,1629.8335
1236,21,female,25.800,0,no,southwest,2007.9450


# 3.데이터 준비

- 전처리 과정을 통해 머신러닝 알고리즘에 사용할 수 있는 형태의 데이터를 준비합니다.

**1) 변수 추가**

In [8]:
# age_grp 변수 추가 (연령대 구하기)
data['age_grp'] = data['age'] // 10 * 10

# 확인
data.head()

,age,sex,bmi,children,smoker,region,charges,age_grp
0,41,female,31.600,0,no,southwest,6186.1270,40
1,30,male,25.460,0,no,northeast,3645.0894,30
2,18,female,30.115,0,no,northeast,21344.8467,10
3,61,female,29.920,3,yes,southeast,30942.1918,60
4,34,female,27.500,1,no,southwest,5003.8530,30


**2) 변수 제거**

In [10]:
# age 변수 제거
data.drop(columns='age', inplace=True) 

# 확인
data.head()

,sex,bmi,children,smoker,region,charges,age_grp
0,female,31.600,0,no,southwest,6186.1270,40
1,male,25.460,0,no,northeast,3645.0894,30
2,female,30.115,0,no,northeast,21344.8467,10
3,female,29.920,3,yes,southeast,30942.1918,60
4,female,27.500,1,no,southwest,5003.8530,30


**3) 가변수화**

In [12]:
# 가변수화
dumm_cols = ['sex', 'smoker', 'region']
data = pd.get_dummies(data, columns=dumm_cols, drop_first=True, dtype=int)

# 확인
data.head()

,bmi,children,charges,age_grp,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,31.600,0,6186.1270,40,0,0,0,0,1
1,25.460,0,3645.0894,30,1,0,0,0,0
2,30.115,0,21344.8467,10,0,0,0,0,0
3,29.920,3,30942.1918,60,0,1,0,1,0
4,27.500,1,5003.8530,30,0,0,0,0,1


**4) x, y 분리**

In [15]:
# x, y 분리
target = 'charges'
x = data.drop(target, axis=1)
y = data.loc[:, target]

**5) 학습용, 평가용 데이터 분리**

In [18]:
# 학습용, 검증용 분리
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=1)

# 4.모델링

- 본격적으로 모델을 선언하고 학습하고 평가하는 과정을 진행합니다.
- 우선 회귀 문제인지 분류 문제인지 명확히 구분합니다.

In [20]:
# 불러오기
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [21]:
# 선언하기
model = DecisionTreeRegressor(max_depth=5)

In [22]:
# 학습하기
model.fit(x_train, y_train)

DecisionTreeRegressor(max_depth=5)

In [24]:
# 예측하기
y_val_pred = model.predict(x_val)

In [26]:
# 평가하기
print('MAE:', mean_absolute_error(y_val, y_val_pred))
print('R2:', r2_score(y_val, y_val_pred))
# MAE: 2848.925847322852
# R2: 0.8488510105762039

MAE: 2741.5587743592178
R2: 0.8592912242361593


# 5.일반화된 성능

- 과연 앞에서 만든 모델의 일반화된 성능은 어떻게 될까요?
- 즉, 고객사에 어느 정도의 성능이 예상된다고 말할 수 있을까요?
- K-Fold CV로 일반화된 성능을 예측합니다.

In [31]:
# 불러오기
from sklearn.model_selection import cross_val_score

# 성능예측
cv_score = cross_val_score(model, x_train, y_train, cv=5)

# 결과
print(cv_score)
print('평균:', cv_score.mean())

[0.81697328 0.8486171  0.79210462 0.82941891 0.81859684]
평균: 0.8211421508158484


# 6. 성능 튜닝

- 앞에서 max_depth 값을 5로 고정함이 아쉽습니다.
- Grid Search로 최적의 하이퍼파라미터를 찾아봅니다.

In [33]:
# 불러오기
from sklearn.model_selection import GridSearchCV

# 파라미터 선언
params = {'max_depth': range(1, 51)}

# 모델 선언
model = GridSearchCV(DecisionTreeRegressor(),
                     params,
                     cv=5,
                     scoring='r2')

In [36]:
# 학습하기
model.fit(x_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(),
             param_grid={'max_depth': range(1, 51)}, scoring='r2')

In [37]:
# 예측 결과 확인
print(model.best_params_)
print(model.best_score_)

{'max_depth': 3}
0.8345716010799604


In [38]:
# 성능 검증
y_val_pred = model.predict(x_val)
print(r2_score(y_val, y_val_pred))

0.8553323704536546


# 7.예측하기

- 최적의 하이퍼파라미터로 학습된 모델로 새로운 데이터에 대해 예측합니다.
- 새로운 데이터에 대해서도 필요한 전처리를 수행합니다.

**1) 데이터 파이프라인**

In [42]:
# 함수 만들기
def data_pipeline(data):
    tmp = data.copy()

    # age_grp 변수 추가
    tmp['age_grp'] = tmp['age'] // 10 * 10

    # age 변수 제거
    tmp.drop(columns='age', inplace=True) 

    # 가변수화
    dumm_cols = ['sex', 'smoker', 'region']
    tmp = pd.get_dummies(tmp, columns=dumm_cols, drop_first=True, dtype=int)

    # 반환
    return tmp

**2) 데이터 불러오기**

In [45]:
# 새로운 데이터 불러오기
path = 'https://raw.githubusercontent.com/Jangrae/csv/master/insurance_test.csv'
new_data = pd.read_csv(path)

# 확인
new_data.head()

,age,sex,bmi,children,smoker,region
0,19,female,27.900,0,yes,southwest
1,18,male,33.770,1,no,southeast
2,28,male,33.000,3,no,southeast
3,33,male,22.705,0,no,northwest
4,32,male,28.880,0,no,northwest


In [46]:
# 데이터 처리
x_test = data_pipeline(new_data)

# 확인
x_test.head()

,bmi,children,age_grp,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,27.900,0,10,0,1,0,0,1
1,33.770,1,10,1,0,0,1,0
2,33.000,3,20,1,0,0,1,0
3,22.705,0,30,1,0,1,0,0
4,28.880,0,30,1,0,1,0,0


**3) 예측하기**

In [50]:
# 예측하기
y_pred = model.predict(x_test)

In [51]:
# 확인
y_pred[:10]

array([18799.46002821,  6489.78888071,  6489.78888071,  3917.77434337,
        3917.77434337,  3917.77434337,  9344.72070988,  6489.78888071,
        6489.78888071, 13214.46218279])

# 8. 파일 제출

- 모델로 예측한 값을 파일에 붙여 엑셀 파일로 저장해 고객사에 제출합니다.

In [55]:
# 예측결과 추가
final = new_data.copy()
final['charges'] = y_pred.round(4)

# 확인
final.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,18799.4600
1,18,male,33.770,1,no,southeast,6489.7889
2,28,male,33.000,3,no,southeast,6489.7889
3,33,male,22.705,0,no,northwest,3917.7743
4,32,male,28.880,0,no,northwest,3917.7743


In [57]:
# 엑셀로 저장
final.to_excel('InsurancePred.xlsx', index=False)

# 9.정리

- 실제 의료비가 지출된 경우 우리 모델의 성능이 확인될 것입니다.
- 우리는 학습용 데이터로 나름 최선을 다했습니다.
- 그래도 뭔가 아쉬움이 남지요?
- 성능이 더 좋은 알고리즘을 배워서 사용할 수 있다면....